In [ ]:
# ============================================================
# BIASCOPE — Notebook 07 (v2, fixed): Multilingual models (XLM-R / MuRIL)
# Full FT vs LoRA | train: en / hi / en+hi | eval: en, hi, gold
# Resume supported: results.csv me jo runs hain unhe skip karta hai
# Fixes: torchao conflict, warmup_ratio (new transformers), results guard
# ============================================================
!pip -q uninstall -y torchao
!pip -q install -U peft transformers accelerate

import gc, inspect, json, time, traceback, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, brier_score_loss)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)
from transformers.utils import logging as hf_logging
from peft import LoraConfig, get_peft_model, TaskType
from google.colab import drive

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
drive.mount("/content/drive")

# ---------------- Config ----------------
RUN_PROFILE = "quick"     # "quick" | "standard" | "full"
PROFILES = {
    "quick":    dict(models=["xlmr"],          seeds=[42]),
    "standard": dict(models=["xlmr", "muril"], seeds=[42]),
    "full":     dict(models=["xlmr", "muril"], seeds=[42, 43, 44]),
}
MODEL_NAMES = {"xlmr": "xlm-roberta-base", "muril": "google/muril-base-cased"}
METHODS = ["full", "lora"]
TRAIN_SETS = ["en", "hi", "en+hi"]
HI_VARIANT = "filtered"          # Hindi training data: "filtered" | "unfiltered"

MAX_LEN, EPOCHS, TRAIN_BS, EVAL_BS = 160, 3, 32, 64
WARMUP, WEIGHT_DECAY = 0.1, 0.01
LR = {"full": 2e-5, "lora": 1e-4}
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1
LORA_TARGET_REGEX = (r".*encoder\.layer\.\d+\.(attention\.self\.(query|key|value)"
                     r"|attention\.output\.dense|intermediate\.dense|output\.dense)")
SAVE_ADAPTERS = True             # LoRA adapters (chhote) save honge, first seed ke liye
MIN_GROUP_N = 40                 # dataset_id breakdown ke liye min samples
GROUP_EVALS = {"en_test", "hi_test_filtered"}

# ---------------- Paths ----------------
BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
V2_DIR = BASE_DIR / "data" / "multilingual" / "mbib_hindi_translation_v2"
GOLD_CANDIDATES = [
    BASE_DIR / "data" / "multilingual" / "gold" / "hindi_gold_llm_labelled.csv",
    Path("/content/hindi_gold_llm_labelled.csv"),
]
OUT_DIR = BASE_DIR / "outputs" / "nb07_multilingual"
MODEL_OUT = BASE_DIR / "models" / "nb07"
for d in (OUT_DIR, MODEL_OUT):
    d.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUT_DIR / "results.csv"
PREDS_PATH = OUT_DIR / "predictions.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 70)
print("BIASCOPE — NOTEBOOK 07 | profile:", RUN_PROFILE, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NO GPU (bahut slow!)")
print("=" * 70)

# ---------------- Data ----------------
def load_en(split):
    df = pd.read_csv(PROCESSED_DIR / f"mbib_{split}.csv")
    out = df[["id", "text", "label", "dataset_id"]].copy()
    out["text"] = out["text"].fillna("").astype(str)
    return out

def load_hi(split, variant="filtered"):
    suffix = "_hi_filtered.csv" if variant == "filtered" else "_hi.csv"
    p = V2_DIR / f"mbib_{split}{suffix}"
    assert p.exists(), f"Missing: {p}"
    df = pd.read_csv(p, encoding="utf-8-sig")
    out = df[["id", "text_hi", "label", "dataset_id"]].rename(columns={"text_hi": "text"})
    out["text"] = out["text"].fillna("").astype(str)
    return out

en_train, en_val, en_test = load_en("train"), load_en("validation"), load_en("test")
hi_train, hi_val = load_hi("train", HI_VARIANT), load_hi("validation", HI_VARIANT)
hi_test_f, hi_test_u = load_hi("test", "filtered"), load_hi("test", "unfiltered")

TRAIN = {
    "en":    (en_train, en_val),
    "hi":    (hi_train, hi_val),
    "en+hi": (pd.concat([en_train, hi_train], ignore_index=True),
              pd.concat([en_val, hi_val], ignore_index=True)),
}

EVAL_SETS = {"en_test": en_test, "hi_test_filtered": hi_test_f, "hi_test_unfiltered": hi_test_u}

gold_path = next((p for p in GOLD_CANDIDATES if p.exists()), None)
if gold_path is None:
    print("\n⚠ hindi_gold_llm_labelled.csv nahi mila — gold evaluation skip hoga.")
else:
    gd = pd.read_csv(gold_path, encoding="utf-8-sig")
    gd = gd[gd["usable"] == 1]
    gold = pd.DataFrame({"id": gd["gold_id"].astype(str), "text": gd["text_hi"].astype(str),
                         "label": gd["bias_label"].astype(int), "dataset_id": "gold",
                         "confidence": gd["confidence"].astype(int)})
    EVAL_SETS["gold_all"] = gold.drop(columns="confidence")
    EVAL_SETS["gold_conf2"] = gold[gold["confidence"] >= 2].drop(columns="confidence")
    print("\nGold loaded from:", gold_path)

print("\nData summary")
for k, (tr, va) in TRAIN.items():
    print(f"  train[{k:5s}] n={len(tr):5d} | val n={len(va):4d} | labels {tr['label'].value_counts().to_dict()}")
for k, df in EVAL_SETS.items():
    print(f"  eval [{k:18s}] n={len(df):4d} | pos_rate={df['label'].mean():.2f}")
print("  test dataset_id counts (en_test):", en_test["dataset_id"].astype(str).value_counts().to_dict())

# ---------------- Dataset / metrics ----------------
class TextDS(torch.utils.data.Dataset):
    def __init__(self, df, tok):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.labels = df["label"].astype(int).tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

def ece_score(y, p, bins=15):
    conf = np.maximum(p, 1 - p)
    correct = ((p >= 0.5).astype(int) == y).astype(float)
    edges = np.linspace(0.5, 1.0, bins + 1)
    edges[-1] = 1.0001
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def compute_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    pred = (p >= 0.5).astype(int)
    out = dict(n=len(y), pos_rate=float(y.mean()),
               acc=accuracy_score(y, pred),
               f1=f1_score(y, pred, zero_division=0),
               macro_f1=f1_score(y, pred, average="macro", zero_division=0),
               precision=precision_score(y, pred, zero_division=0),
               recall=recall_score(y, pred, zero_division=0),
               ece=ece_score(y, p), brier=brier_score_loss(y, p))
    out["auc"] = roc_auc_score(y, p) if len(set(y)) == 2 else np.nan
    return out

def predict_probs(trainer, ds):
    logits = trainer.predict(ds).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e[:, 1] / e.sum(axis=1)

def build_model(model_key, method):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAMES[model_key], num_labels=2)
    if method == "lora":
        cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA,
                         lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_REGEX, bias="none")
        model = get_peft_model(model, cfg)
    return model

# ---------------- One run ----------------
def run_one(model_key, method, train_name, seed, tok, first_seed):
    run_id = f"{model_key}|{method}|{train_name}|s{seed}"
    train_df, val_df = TRAIN[train_name]
    set_seed(seed)
    model = build_model(model_key, method)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    steps_per_epoch = int(np.ceil(len(train_df) / TRAIN_BS))
    warmup_steps = int(WARMUP * steps_per_epoch * EPOCHS)
    ta_kwargs = dict(
        output_dir=f"/content/ckpt/{model_key}_{method}_{train_name.replace('+', '_')}_{seed}",
        num_train_epochs=EPOCHS, per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS, learning_rate=LR[method],
        weight_decay=WEIGHT_DECAY, warmup_steps=warmup_steps,
        fp16=torch.cuda.is_available(), save_strategy="no", logging_strategy="no",
        report_to="none", disable_tqdm=True, seed=seed, data_seed=seed,
        dataloader_num_workers=2, remove_unused_columns=False, label_names=["labels"],
    )
    supported = set(inspect.signature(TrainingArguments.__init__).parameters)
    dropped = [k for k in ta_kwargs if k not in supported]
    if dropped:
        print("   (is version me ye TrainingArguments nahi hain, skip:", dropped, ")")
    args = TrainingArguments(**{k: v for k, v in ta_kwargs.items() if k in supported})

    collator = DataCollatorWithPadding(tok)
    train_ds = TextDS(train_df, tok)
    try:
        trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                          data_collator=collator, processing_class=tok)
    except TypeError:
        trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                          data_collator=collator, tokenizer=tok)

    t0 = time.time()
    trainer.train()
    train_sec = time.time() - t0

    eval_all = dict(EVAL_SETS)
    eval_all["val"] = val_df
    base = dict(run_id=run_id, model=model_key, method=method, train_set=train_name, seed=seed,
                trainable_params=trainable, total_params=total, train_seconds=round(train_sec, 1))
    rows, pred_rows = [], []
    for name, df in eval_all.items():
        df = df.reset_index(drop=True)
        p = predict_probs(trainer, TextDS(df, tok))
        y = df["label"].values
        rows.append({**base, "eval_set": name, "group": "all", **compute_metrics(y, p)})
        pred_rows.append(pd.DataFrame({"run_id": run_id, "eval_set": name,
                                       "row_id": df["id"].astype(str), "y": y, "p": p}))
        if name in GROUP_EVALS:
            for gid, g in df.groupby(df["dataset_id"].astype(str)):
                if len(g) >= MIN_GROUP_N:
                    idx = g.index.values
                    rows.append({**base, "eval_set": name, "group": f"ds={gid}",
                                 **compute_metrics(y[idx], p[idx])})

    val_row = next(r for r in rows if r["eval_set"] == "val" and r["group"] == "all")
    if val_row["auc"] < 0.6:
        print(f"   ⚠ {run_id}: val AUC {val_row['auc']:.3f} — run collapse hua lag raha hai")

    pd.DataFrame(rows).to_csv(RESULTS_PATH, mode="a", header=not RESULTS_PATH.exists(), index=False)
    pd.concat(pred_rows).to_csv(PREDS_PATH, mode="a", header=not PREDS_PATH.exists(), index=False)

    if SAVE_ADAPTERS and method == "lora" and first_seed:
        path = MODEL_OUT / f"{model_key}_lora_{train_name.replace('+', '_')}_s{seed}"
        trainer.model.save_pretrained(str(path))
        tok.save_pretrained(str(path))

    del trainer, model, train_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return val_row, train_sec

# ---------------- Plan + run ----------------
prof = PROFILES[RUN_PROFILE]
done = set(pd.read_csv(RESULTS_PATH)["run_id"]) if RESULTS_PATH.exists() else set()
plan = [(m, meth, ts, s) for s in prof["seeds"] for m in prof["models"]
        for meth in METHODS for ts in TRAIN_SETS
        if f"{m}|{meth}|{ts}|s{s}" not in done]
print(f"\nPlanned runs: {len(plan)} (already done: {len(done)})")

json.dump(dict(profile=RUN_PROFILE, models=prof["models"], seeds=prof["seeds"], methods=METHODS,
               train_sets=TRAIN_SETS, hi_variant=HI_VARIANT, max_len=MAX_LEN, epochs=EPOCHS,
               train_bs=TRAIN_BS, lr=LR, lora=dict(r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT,
               targets=LORA_TARGET_REGEX)), open(OUT_DIR / "config.json", "w"), indent=2)

tokenizers, failed, times = {}, [], []
for i, (m, meth, ts, s) in enumerate(plan, 1):
    run_id = f"{m}|{meth}|{ts}|s{s}"
    try:
        if m not in tokenizers:
            tokenizers[m] = AutoTokenizer.from_pretrained(MODEL_NAMES[m])
        eta = (np.mean(times) * (len(plan) - i + 1) / 60) if times else float("nan")
        print(f"\n[{i}/{len(plan)}] {run_id} | ETA ~{eta:.0f} min")
        val_row, sec = run_one(m, meth, ts, s, tokenizers[m], first_seed=(s == prof["seeds"][0]))
        times.append(sec)
        print(f"   done in {sec / 60:.1f} min | val F1={val_row['f1']:.3f} AUC={val_row['auc']:.3f}")
    except Exception as e:
        print(f"   ❌ FAILED {run_id}: {repr(e)[:300]}")
        if not failed:
            print("   --- traceback (first failure) ---")
            print(traceback.format_exc()[-1200:])
        failed.append(run_id)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if failed:
    print("\nFailed runs:", failed)

assert RESULTS_PATH.exists(), "Koi run complete nahi hua — upar ke FAILED errors/traceback dekho."

# ============================================================
# SUMMARY
# ============================================================
res = pd.read_csv(RESULTS_PATH)
res_f = res[res["group"] == "all"]
key_sets = [s for s in ["en_test", "hi_test_filtered", "gold_conf2"] if s in res_f["eval_set"].unique()]

def fmt_ms(x):
    return f"{x.mean():.3f}" + (f"±{x.std():.3f}" if len(x) > 1 else "")

print("\n" + "=" * 70 + "\nMAIN RESULTS (mean±std over seeds)\n" + "=" * 70)
sub = res_f[res_f["eval_set"].isin(key_sets)]
for metric in ["f1", "auc", "ece"]:
    print(f"\n--- {metric.upper()} ---")
    tab = (sub.groupby(["model", "method", "train_set", "eval_set"])[metric].apply(fmt_ms)
              .unstack("eval_set")[key_sets])
    display(tab)

print("\n--- LoRA minus Full (mean F1) ---")
piv = sub.groupby(["model", "train_set", "eval_set", "method"])["f1"].mean().unstack("method")
if {"full", "lora"} <= set(piv.columns):
    piv["lora_minus_full"] = (piv["lora"] - piv["full"]).round(3)
    display(piv.round(3))

print("\n--- Cross-lingual transfer (mean F1; hi_minus_en < 0 => Hindi test me drop) ---")
t = sub.groupby(["model", "method", "train_set", "eval_set"])["f1"].mean().unstack("eval_set")
if {"en_test", "hi_test_filtered"} <= set(t.columns):
    t["hi_minus_en"] = (t["hi_test_filtered"] - t["en_test"]).round(3)
display(t.round(3))

bd = res[res["group"].str.startswith("ds=") & res["eval_set"].isin(GROUP_EVALS)]
if len(bd):
    print("\n--- F1 by dataset_id (news-like vs Reddit-like) ---")
    display(bd.groupby(["model", "method", "train_set", "eval_set", "group"])["f1"].mean()
              .unstack(["eval_set", "group"]).round(3))
    print("Group sizes:", bd.drop_duplicates(["eval_set", "group"])[["eval_set", "group", "n", "pos_rate"]]
          .round(2).to_dict("records"))

# ---- chart ----
agg_m = sub.groupby(["model", "method", "train_set", "eval_set"])["f1"].mean().unstack("eval_set")[key_sets]
agg_m.index = [" | ".join(i) for i in agg_m.index]
ax = agg_m.plot(kind="bar", figsize=(13, 5))
ax.set_ylabel("F1 (positive class)")
ax.set_title("BIASCOPE NB07 — F1 by config and eval set")
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "nb07_f1_summary.png", dpi=150)
plt.show()

print("\nSaved:", RESULTS_PATH, "|", PREDS_PATH, "|", OUT_DIR / "nb07_f1_summary.png")
print("""
REPORT ME YE LIKHNA:
- Hindi training labels MBIB se inherit hue hain (MT: NLLB-200); koi human verification nahi.
- gold_* LLM-annotated hai (single pass), enriched sample, mostly sensationalism; gold_conf2 primary hai.
- Test sets chhote hain (SE ~1.5% on ~850 rows; gold_conf2 ~240 rows), isliye <2-3 F1 points ke gap ko
  'noise ke andar' maano jab tak 3 seeds me consistent na ho.
- MBIB test ka bada hissa Reddit-style text hai; dataset_id breakdown wala table zaroor report karo.
NEXT: Notebook 08 — calibration (ECE, temperature scaling) + Integrated Gradients + robustness.
""")

In [ ]:
# ============================================================
# BIASCOPE — Notebook 07b: BABE (expert-labelled news bias) + Hindi translation
# Experiments: train on mbib | babe | mbib+babe (each en+hi), full FT vs LoRA
# Eval: BABE test (en/hi), MBIB test (en/hi), native Hindi gold (LLM-labelled)
# Resume supported (translation cache + results.csv)
# ============================================================
!pip -q uninstall -y torchao
!pip -q install -U peft transformers accelerate sentencepiece

import gc, inspect, json, math, re, time, traceback, warnings
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, brier_score_loss)
from sklearn.model_selection import GroupShuffleSplit
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)
from transformers.utils import logging as hf_logging
from peft import LoraConfig, get_peft_model, TaskType
from google.colab import drive

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
drive.mount("/content/drive")

# ---------------- Config ----------------
RUN_PROFILE = "quick"     # "quick" (xlmr, 1 seed) | "standard" (+muril) | "full" (+3 seeds)
PROFILES = {
    "quick":    dict(models=["xlmr"],          seeds=[42]),
    "standard": dict(models=["xlmr", "muril"], seeds=[42]),
    "full":     dict(models=["xlmr", "muril"], seeds=[42, 43, 44]),
}
MODEL_NAMES = {"xlmr": "xlm-roberta-base", "muril": "google/muril-base-cased"}
METHODS = ["full", "lora"]
TRAIN_SETS = ["mbib", "babe", "mbib+babe"]
BABE_UPSAMPLE = 2          # mbib+babe me BABE rows kitni baar repeat hon

MT_MODEL = "facebook/nllb-200-distilled-1.3B"
SRC_LANG, TGT_LANG = "eng_Latn", "hin_Deva"
MT_BATCH, MT_MAX_WORDS, MT_MAX_NEW = 24, 30, 200
MT_BEAMS, MT_NO_REPEAT, MT_REP_PEN = 4, 4, 1.1

MAX_LEN, EPOCHS, TRAIN_BS, EVAL_BS = 160, 3, 32, 64
WARMUP, WEIGHT_DECAY = 0.1, 0.01
LR = {"full": 2e-5, "lora": 1e-4}
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1
LORA_TARGET_REGEX = (r".*encoder\.layer\.\d+\.(attention\.self\.(query|key|value)"
                     r"|attention\.output\.dense|intermediate\.dense|output\.dense)")
MIN_GROUP_N = 40
GROUP_EVALS = {"babe_test_en", "mbib_en_test"}
SEED = 42

# ---------------- Paths ----------------
BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
V2_DIR = BASE_DIR / "data" / "multilingual" / "mbib_hindi_translation_v2"
BABE_DIR = BASE_DIR / "data" / "multilingual" / "babe"
GOLD_CANDIDATES = [BASE_DIR / "data" / "multilingual" / "gold" / "hindi_gold_llm_labelled.csv",
                   Path("/content/hindi_gold_llm_labelled.csv")]
OUT_DIR = BASE_DIR / "outputs" / "nb07b_babe"
for d in (BABE_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
RESULTS_PATH, PREDS_PATH = OUT_DIR / "results.csv", OUT_DIR / "predictions.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(SEED)

print("=" * 70)
print("BIASCOPE — NOTEBOOK 07b | profile:", RUN_PROFILE, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NO GPU (bahut slow!)")
print("=" * 70)

# ============================================================
# 1. LOAD BABE
# ============================================================
def load_babe():
    from datasets import load_dataset
    errors = []
    for kw in [dict(path="mediabiasgroup/BABE"),
               dict(path="mediabiasgroup/BABE", revision="refs/convert/parquet")]:
        try:
            d = load_dataset(**kw)
            print("BABE loaded with:", kw, "| splits:", {k: len(v) for k, v in d.items()})
            return {k: v.to_pandas() for k, v in d.items()}
        except Exception as e:
            errors.append((kw, repr(e)[:200]))
    from huggingface_hub import HfApi, hf_hub_download
    for rev in [None, "refs/convert/parquet"]:
        try:
            files = HfApi().list_repo_files("mediabiasgroup/BABE", repo_type="dataset", revision=rev)
            pq = [f for f in files if f.endswith(".parquet")]
            out = {}
            for split in ["train", "test"]:
                pick = [f for f in pq if split in f.lower()]
                if pick:
                    out[split] = pd.concat([pd.read_parquet(hf_hub_download(
                        "mediabiasgroup/BABE", f, repo_type="dataset", revision=rev)) for f in sorted(pick)])
            if {"train", "test"} <= set(out):
                print("BABE loaded via parquet files (rev=%s)" % rev)
                return out
        except Exception as e:
            errors.append((rev, repr(e)[:200]))
    print("BABE load errors:", errors)
    raise RuntimeError("BABE load nahi hua — upar ka error output mujhe bhej do.")

babe = load_babe()
tr_all, te_all = babe["train"].reset_index(drop=True), babe["test"].reset_index(drop=True)
print("BABE columns:", tr_all.columns.tolist())
assert {"text", "label"} <= set(tr_all.columns), "BABE me 'text'/'label' columns nahi mile"
for name, d in (("train", tr_all), ("test", te_all)):
    d["text"] = d["text"].fillna("").astype(str)
    d["label"] = d["label"].astype(int)
    if "uuid" not in d.columns:
        d["uuid"] = [f"{name}_{i}" for i in range(len(d))]
    d["uuid"] = d["uuid"].astype(str)
    d["lean"] = d["type"].fillna("none").astype(str) if "type" in d.columns else "none"
    d["group_id"] = d["news_link"].fillna(d["uuid"]).astype(str) if "news_link" in d.columns else d["uuid"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
tr_idx, va_idx = next(gss.split(tr_all, groups=tr_all["group_id"]))
babe_split = pd.concat([
    tr_all.iloc[tr_idx].assign(split="train"),
    tr_all.iloc[va_idx].assign(split="val"),
    te_all.assign(split="test"),
], ignore_index=True)
assert babe_split["uuid"].is_unique, "BABE uuid unique nahi"
print("BABE splits:", babe_split.groupby("split")["label"].agg(["size", "mean"]).round(3).to_dict("index"))

# ============================================================
# 2. TRANSLATE BABE -> HINDI (NLLB, sentence-level, cached)
# ============================================================
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

def segment_text(text, max_words=MT_MAX_WORDS):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text or text.lower() == "nan":
        return []
    segs = []
    for s in _SENT_SPLIT.split(text):
        w = s.split()
        if not w:
            continue
        if len(w) <= max_words:
            segs.append(" ".join(w))
        else:
            n = math.ceil(len(w) / max_words)
            size = math.ceil(len(w) / n)
            segs += [" ".join(w[i:i + size]) for i in range(0, len(w), size)]
    return segs

DEV = r"\u0900-\u097F"
def letter_ratios(t):
    dev = len(re.findall(rf"[{DEV}]", str(t)))
    lat = len(re.findall(r"[A-Za-z]", str(t)))
    tot = dev + lat
    return (dev / tot, lat / tot) if tot else (0.0, 0.0)
def uniq_ratio(t):
    w = str(t).split()
    return len(set(w)) / len(w) if w else 1.0
def max_ngram_repeat(t, n=3):
    w = str(t).split()
    g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return max(Counter(g).values()) if g else 0

def quality_flags(src, hi):
    n_en = src.map(lambda t: len(str(t).split()))
    n_hi = hi.map(lambda t: len(str(t).split()))
    ratio = (n_hi + 1) / (n_en + 1)
    f = pd.DataFrame({
        "too_short": (n_hi < 3) & (n_en >= 6),
        "len_ratio": (ratio < 0.4) | (ratio > 2.2),
        "repeat": (n_hi >= 12) & (hi.map(uniq_ratio).lt(0.45) | hi.map(max_ngram_repeat).ge(3)),
        "untranslated": hi.map(lambda t: letter_ratios(t)[1]) > 0.35,
    })
    f["auto_flag"] = f.any(axis=1)
    return f

def translate_babe(df):
    out_path = BABE_DIR / "babe_hi.csv"
    cache = {}
    if out_path.exists():
        ex = pd.read_csv(out_path, encoding="utf-8-sig")
        cache = dict(zip(ex["uuid"].astype(str), ex["text_hi"].fillna("").astype(str)))
    todo = df[~df["uuid"].isin([k for k, v in cache.items() if v.strip()])]
    print(f"BABE translation: cached={len(cache)} | to translate={len(todo)}")
    if len(todo):
        tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang=SRC_LANG)
        dt = torch.float16 if DEVICE == "cuda" else torch.float32
        mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL, torch_dtype=dt).to(DEVICE).eval()
        tgt = tok.convert_tokens_to_ids(TGT_LANG)
        assert tgt != tok.unk_token_id, f"{TGT_LANG} token nahi mila"

        @torch.no_grad()
        def _batch(b):
            try:
                enc = tok(b, return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)
                gen = mt.generate(**enc, forced_bos_token_id=tgt, num_beams=MT_BEAMS, max_new_tokens=MT_MAX_NEW,
                                  no_repeat_ngram_size=MT_NO_REPEAT, repetition_penalty=MT_REP_PEN,
                                  early_stopping=True)
                return [t.strip() for t in tok.batch_decode(gen, skip_special_tokens=True)]
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if len(b) == 1:
                    raise
                h = len(b) // 2
                return _batch(b[:h]) + _batch(b[h:])

        def translate_texts(texts):
            segs, owners = [], []
            for r, t in enumerate(texts):
                for s in segment_text(t):
                    segs.append(s); owners.append(r)
            order = sorted(range(len(segs)), key=lambda i: len(segs[i]))
            out = [None] * len(segs)
            for b in range(0, len(order), MT_BATCH):
                idx = order[b:b + MT_BATCH]
                for i, t in zip(idx, _batch([segs[i] for i in idx])):
                    out[i] = t
            rows = [[] for _ in texts]
            for r, o in zip(owners, out):
                rows[r].append(o)
            return [" ".join(x) for x in rows]

        t0 = time.time()
        CH = 400
        for c in range(0, len(todo), CH):
            part = todo.iloc[c:c + CH]
            for u, h in zip(part["uuid"], translate_texts(part["text"].tolist())):
                cache[u] = h
            snap = df[["uuid", "split", "text", "label"]].copy()
            snap["text_hi"] = snap["uuid"].map(cache).fillna("")
            snap.to_csv(out_path, index=False, encoding="utf-8-sig")
            print(f"  [{min(c + CH, len(todo))}/{len(todo)}] elapsed={(time.time() - t0) / 60:.1f} min")
        del mt, tok
        gc.collect(); torch.cuda.empty_cache()
    res = df[["uuid", "split", "text", "label"]].copy()
    res["text_hi"] = res["uuid"].map(cache).fillna("")
    res.to_csv(out_path, index=False, encoding="utf-8-sig")
    return res

babe_hi = translate_babe(babe_split)
flags = quality_flags(babe_hi["text"], babe_hi["text_hi"])
babe_hi["auto_flag"] = flags["auto_flag"].values
print(f"\nBABE Hindi flagged: {babe_hi['auto_flag'].mean():.1%} | by split:",
      babe_hi.groupby("split")["auto_flag"].mean().round(3).to_dict())
print("Flag rate by label:", babe_hi.groupby("label")["auto_flag"].mean().round(3).to_dict())
pd.set_option("display.max_colwidth", 150)
print("\nSample BABE translations:")
display(babe_hi[["text", "text_hi", "label"]].sample(6, random_state=SEED))

# ============================================================
# 3. DATA FOR EXPERIMENTS
# ============================================================
def load_en(split):
    df = pd.read_csv(PROCESSED_DIR / f"mbib_{split}.csv")
    return pd.DataFrame({"id": df["id"].astype(str), "text": df["text"].fillna("").astype(str),
                         "label": df["label"].astype(int), "group_key": df["dataset_id"].astype(str)})

def load_hi(split, variant="filtered"):
    suffix = "_hi_filtered.csv" if variant == "filtered" else "_hi.csv"
    df = pd.read_csv(V2_DIR / f"mbib_{split}{suffix}", encoding="utf-8-sig")
    return pd.DataFrame({"id": df["id"].astype(str), "text": df["text_hi"].fillna("").astype(str),
                         "label": df["label"].astype(int), "group_key": df["dataset_id"].astype(str)})

mb = {s: pd.concat([load_en(s), load_hi(s)], ignore_index=True) for s in ["train", "validation"]}
mbib_en_test, mbib_hi_test = load_en("test"), load_hi("test")

def babe_part(split, lang):
    d = babe_hi[babe_hi["split"] == split].merge(babe_split[["uuid", "lean"]], on="uuid")
    if lang == "hi":
        d = d[~d["auto_flag"]]
    col = "text" if lang == "en" else "text_hi"
    return pd.DataFrame({"id": d["uuid"] + f"_{lang}", "text": d[col].astype(str),
                         "label": d["label"].astype(int), "group_key": d["lean"]})

babe_tr = pd.concat([babe_part("train", "en"), babe_part("train", "hi")], ignore_index=True)
babe_va = pd.concat([babe_part("val", "en"), babe_part("val", "hi")], ignore_index=True)

TRAIN = {
    "mbib": (mb["train"], mb["validation"]),
    "babe": (babe_tr, babe_va),
    "mbib+babe": (pd.concat([mb["train"]] + [babe_tr] * BABE_UPSAMPLE, ignore_index=True),
                  pd.concat([mb["validation"], babe_va], ignore_index=True)),
}
EVAL_SETS = {"mbib_en_test": mbib_en_test, "mbib_hi_test": mbib_hi_test,
             "babe_test_en": babe_part("test", "en"), "babe_test_hi": babe_part("test", "hi")}

gold_path = next((p for p in GOLD_CANDIDATES if p.exists()), None)
if gold_path is None:
    print("\n⚠ hindi_gold_llm_labelled.csv nahi mila — gold evaluation skip hoga.")
else:
    gd = pd.read_csv(gold_path, encoding="utf-8-sig")
    gd = gd[gd["usable"] == 1]
    gold = pd.DataFrame({"id": gd["gold_id"].astype(str), "text": gd["text_hi"].astype(str),
                         "label": gd["bias_label"].astype(int), "group_key": "gold",
                         "confidence": gd["confidence"].astype(int)})
    EVAL_SETS["gold_all"] = gold.drop(columns="confidence")
    EVAL_SETS["gold_conf2"] = gold[gold["confidence"] >= 2].drop(columns="confidence")

print("\nData summary")
for k, (tr, va) in TRAIN.items():
    print(f"  train[{k:10s}] n={len(tr):6d} | val n={len(va):5d} | pos_rate={tr['label'].mean():.2f}")
for k, df in EVAL_SETS.items():
    print(f"  eval [{k:13s}] n={len(df):5d} | pos_rate={df['label'].mean():.2f}")

# ============================================================
# 4. TRAINING / EVALUATION (same recipe as NB07)
# ============================================================
class TextDS(torch.utils.data.Dataset):
    def __init__(self, df, tok):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.labels = df["label"].astype(int).tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

def ece_score(y, p, bins=15):
    conf = np.maximum(p, 1 - p)
    correct = ((p >= 0.5).astype(int) == y).astype(float)
    edges = np.linspace(0.5, 1.0, bins + 1); edges[-1] = 1.0001
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def compute_metrics(y, p):
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    pred = (p >= 0.5).astype(int)
    out = dict(n=len(y), pos_rate=float(y.mean()), acc=accuracy_score(y, pred),
               f1=f1_score(y, pred, zero_division=0),
               macro_f1=f1_score(y, pred, average="macro", zero_division=0),
               precision=precision_score(y, pred, zero_division=0),
               recall=recall_score(y, pred, zero_division=0),
               ece=ece_score(y, p), brier=brier_score_loss(y, p))
    out["auc"] = roc_auc_score(y, p) if len(set(y)) == 2 else np.nan
    return out

def predict_probs(trainer, ds):
    logits = trainer.predict(ds).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e[:, 1] / e.sum(axis=1)

def build_model(model_key, method):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAMES[model_key], num_labels=2)
    if method == "lora":
        cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA,
                         lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_REGEX, bias="none")
        model = get_peft_model(model, cfg)
    return model

def run_one(model_key, method, train_name, seed, tok):
    run_id = f"{model_key}|{method}|{train_name}|s{seed}"
    train_df, val_df = TRAIN[train_name]
    set_seed(seed)
    model = build_model(model_key, method)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    steps = int(np.ceil(len(train_df) / TRAIN_BS))
    ta = dict(output_dir=f"/content/ckpt/{model_key}_{method}_{train_name.replace('+', '_')}_{seed}",
              num_train_epochs=EPOCHS, per_device_train_batch_size=TRAIN_BS,
              per_device_eval_batch_size=EVAL_BS, learning_rate=LR[method],
              weight_decay=WEIGHT_DECAY, warmup_steps=int(WARMUP * steps * EPOCHS),
              fp16=torch.cuda.is_available(), save_strategy="no", logging_strategy="no",
              report_to="none", disable_tqdm=True, seed=seed, data_seed=seed,
              dataloader_num_workers=2, remove_unused_columns=False, label_names=["labels"])
    supported = set(inspect.signature(TrainingArguments.__init__).parameters)
    args = TrainingArguments(**{k: v for k, v in ta.items() if k in supported})
    collator, train_ds = DataCollatorWithPadding(tok), TextDS(train_df, tok)
    try:
        trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                          data_collator=collator, processing_class=tok)
    except TypeError:
        trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                          data_collator=collator, tokenizer=tok)
    t0 = time.time()
    trainer.train()
    sec = time.time() - t0

    eval_all = dict(EVAL_SETS); eval_all["val"] = val_df
    base = dict(run_id=run_id, model=model_key, method=method, train_set=train_name, seed=seed,
                trainable_params=trainable, total_params=total, train_seconds=round(sec, 1))
    rows, preds = [], []
    for name, df in eval_all.items():
        df = df.reset_index(drop=True)
        p = predict_probs(trainer, TextDS(df, tok)); y = df["label"].values
        rows.append({**base, "eval_set": name, "group": "all", **compute_metrics(y, p)})
        preds.append(pd.DataFrame({"run_id": run_id, "eval_set": name, "row_id": df["id"].astype(str), "y": y, "p": p}))
        if name in GROUP_EVALS:
            for gid, g in df.groupby(df["group_key"].astype(str)):
                if len(g) >= MIN_GROUP_N:
                    rows.append({**base, "eval_set": name, "group": f"g={gid}",
                                 **compute_metrics(y[g.index.values], p[g.index.values])})
    val_row = next(r for r in rows if r["eval_set"] == "val" and r["group"] == "all")
    if val_row["auc"] < 0.6:
        print(f"   ⚠ {run_id}: val AUC {val_row['auc']:.3f} — run collapse hua lag raha hai")
    pd.DataFrame(rows).to_csv(RESULTS_PATH, mode="a", header=not RESULTS_PATH.exists(), index=False)
    pd.concat(preds).to_csv(PREDS_PATH, mode="a", header=not PREDS_PATH.exists(), index=False)
    del trainer, model, train_ds
    gc.collect(); torch.cuda.empty_cache()
    return val_row, sec

prof = PROFILES[RUN_PROFILE]
done = set(pd.read_csv(RESULTS_PATH)["run_id"]) if RESULTS_PATH.exists() else set()
plan = [(m, meth, ts, s) for s in prof["seeds"] for m in prof["models"]
        for meth in METHODS for ts in TRAIN_SETS if f"{m}|{meth}|{ts}|s{s}" not in done]
print(f"\nPlanned runs: {len(plan)} (already done: {len(done)})")
json.dump(dict(profile=RUN_PROFILE, models=prof["models"], seeds=prof["seeds"], methods=METHODS,
               train_sets=TRAIN_SETS, babe_upsample=BABE_UPSAMPLE, mt_model=MT_MODEL, max_len=MAX_LEN,
               epochs=EPOCHS, lr=LR, lora=dict(r=LORA_R, alpha=LORA_ALPHA, regex=LORA_TARGET_REGEX)),
          open(OUT_DIR / "config.json", "w"), indent=2)

tokenizers, failed, times = {}, [], []
for i, (m, meth, ts, s) in enumerate(plan, 1):
    run_id = f"{m}|{meth}|{ts}|s{s}"
    try:
        if m not in tokenizers:
            tokenizers[m] = AutoTokenizer.from_pretrained(MODEL_NAMES[m])
        eta = (np.mean(times) * (len(plan) - i + 1) / 60) if times else float("nan")
        print(f"\n[{i}/{len(plan)}] {run_id} | ETA ~{eta:.0f} min")
        val_row, sec = run_one(m, meth, ts, s, tokenizers[m])
        times.append(sec)
        print(f"   done in {sec / 60:.1f} min | val F1={val_row['f1']:.3f} AUC={val_row['auc']:.3f}")
    except Exception as e:
        print(f"   ❌ FAILED {run_id}: {repr(e)[:300]}")
        if not failed:
            print(traceback.format_exc()[-1200:])
        failed.append(run_id)
        gc.collect(); torch.cuda.empty_cache()
if failed:
    print("\nFailed runs:", failed)
assert RESULTS_PATH.exists(), "Koi run complete nahi hua — upar ke errors dekho."

# ============================================================
# 5. SUMMARY
# ============================================================
res = pd.read_csv(RESULTS_PATH)
res_f = res[res["group"] == "all"]
KEY = [s for s in ["mbib_en_test", "mbib_hi_test", "babe_test_en", "babe_test_hi", "gold_conf2"]
       if s in set(res_f["eval_set"])]
sub = res_f[res_f["eval_set"].isin(KEY)]
fmt = lambda x: f"{x.mean():.3f}" + (f"±{x.std():.3f}" if len(x) > 1 else "")

print("\n" + "=" * 70 + "\nMAIN RESULTS (mean±std over seeds)\n" + "=" * 70)
for metric in ["auc", "f1", "ece"]:
    print(f"\n--- {metric.upper()} ---")
    display(sub.groupby(["model", "method", "train_set", "eval_set"])[metric].apply(fmt)
               .unstack("eval_set")[KEY])

for metric in ["auc", "f1"]:
    print(f"\n--- BABE effect on {metric.upper()} (mbib+babe minus mbib; babe minus mbib) ---")
    piv = sub.groupby(["model", "method", "eval_set", "train_set"])[metric].mean().unstack("train_set")
    if {"mbib", "babe", "mbib+babe"} <= set(piv.columns):
        piv["mbib+babe - mbib"] = (piv["mbib+babe"] - piv["mbib"]).round(3)
        piv["babe - mbib"] = (piv["babe"] - piv["mbib"]).round(3)
    display(piv.round(3))

print("\n--- LoRA minus Full (mean AUC) ---")
lp = sub.groupby(["model", "train_set", "eval_set", "method"])["auc"].mean().unstack("method")
if {"full", "lora"} <= set(lp.columns):
    lp["lora - full"] = (lp["lora"] - lp["full"]).round(3)
display(lp.round(3))

grp = res[res["group"].str.startswith("g=") & res["eval_set"].isin(GROUP_EVALS)]
if len(grp):
    print("\n--- AUC by group (BABE: political lean; MBIB: dataset_id) ---")
    display(grp.groupby(["model", "method", "train_set", "eval_set", "group"])["auc"].mean()
               .unstack(["eval_set", "group"]).round(3))
    print("Group sizes:", grp.drop_duplicates(["eval_set", "group"])[["eval_set", "group", "n", "pos_rate"]]
          .round(2).to_dict("records"))

agg = sub.groupby(["model", "method", "train_set", "eval_set"])["auc"].mean().unstack("eval_set")[KEY]
agg.index = [" | ".join(i) for i in agg.index]
ax = agg.plot(kind="bar", figsize=(13, 5))
ax.set_ylabel("AUC"); ax.set_title("BIASCOPE NB07b — AUC by config and eval set")
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
plt.xticks(rotation=35, ha="right"); plt.tight_layout()
plt.savefig(OUT_DIR / "nb07b_auc_summary.png", dpi=150); plt.show()

print("\nSaved:", RESULTS_PATH, "|", PREDS_PATH, "|", BABE_DIR / "babe_hi.csv")
print("""
REPORT ME YE LIKHNA:
- BABE = expert-labelled news sentences (en). Hindi version = NLLB-200 translation, labels inherited, no human check.
- gold_* = native Hindi, LLM-annotated (single pass), enriched sample; AUC primary metric.
- BABE test n=1000, gold_conf2 n~240: chhote gaps (<~0.02 AUC) ko noise maano jab tak 3 seeds consistent na ho.
- LICENSE: MBIB (CC BY-NC-ND 4.0) aur BABE ke translated versions ko publicly redistribute mat karna
  (research use ke liye private rakho; BABE ka license dataset card pe khud check kar lena).
NEXT: Notebook 08 — calibration (temperature scaling), Integrated Gradients, robustness.
""")

In [ ]:
# ============================================================
# BIASCOPE — Notebook 07c: Final model (mbib+babe) — seeds + save adapters
# Requires: 07b cell already run in THIS session (uses its data/helpers).
# ============================================================
assert "TRAIN" in globals() and "build_model" in globals(), \
    "Pehle 07b cell chalao (same session). Translation cached hai, setup ~2 min."

FINAL_DIR = BASE_DIR / "models" / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
R07C, P07C = OUT_DIR / "results_07c.csv", OUT_DIR / "predictions_07c.csv"

RUNS = ([("xlmr", "lora", "mbib+babe", s) for s in (42, 43, 44)]
        + [("xlmr", "full", "mbib+babe", s) for s in (43, 44)]
        + [("muril", "lora", "mbib+babe", 42)])

def train_and_save(model_key, method, train_name, seed):
    run_id = f"{model_key}|{method}|{train_name}|s{seed}"
    tok = AutoTokenizer.from_pretrained(MODEL_NAMES[model_key])
    train_df, val_df = TRAIN[train_name]
    set_seed(seed)
    model = build_model(model_key, method)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    steps = int(np.ceil(len(train_df) / TRAIN_BS))
    ta = dict(output_dir=f"/content/ckpt/{model_key}_{method}_{seed}_07c",
              num_train_epochs=EPOCHS, per_device_train_batch_size=TRAIN_BS,
              per_device_eval_batch_size=EVAL_BS, learning_rate=LR[method],
              weight_decay=WEIGHT_DECAY, warmup_steps=int(WARMUP * steps * EPOCHS),
              fp16=torch.cuda.is_available(), save_strategy="no", logging_strategy="no",
              report_to="none", disable_tqdm=True, seed=seed, data_seed=seed,
              dataloader_num_workers=2, remove_unused_columns=False, label_names=["labels"])
    supported = set(inspect.signature(TrainingArguments.__init__).parameters)
    args = TrainingArguments(**{k: v for k, v in ta.items() if k in supported})
    collator, train_ds = DataCollatorWithPadding(tok), TextDS(train_df, tok)
    try:
        trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                          data_collator=collator, processing_class=tok)
    except TypeError:
        trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                          data_collator=collator, tokenizer=tok)
    t0 = time.time()
    trainer.train()
    sec = time.time() - t0

    eval_all = dict(EVAL_SETS); eval_all["val"] = val_df
    base = dict(run_id=run_id, model=model_key, method=method, train_set=train_name, seed=seed,
                trainable_params=trainable, total_params=total, train_seconds=round(sec, 1))
    rows, preds = [], []
    for name, df in eval_all.items():
        df = df.reset_index(drop=True)
        p = predict_probs(trainer, TextDS(df, tok)); y = df["label"].values
        rows.append({**base, "eval_set": name, "group": "all", **compute_metrics(y, p)})
        preds.append(pd.DataFrame({"run_id": run_id, "eval_set": name,
                                   "row_id": df["id"].astype(str), "y": y, "p": p}))
    pd.DataFrame(rows).to_csv(R07C, mode="a", header=not R07C.exists(), index=False)
    pd.concat(preds).to_csv(P07C, mode="a", header=not P07C.exists(), index=False)

    saved = None
    if method == "lora":
        saved = FINAL_DIR / f"{model_key}_lora_{train_name.replace('+', '_')}_s{seed}"
        trainer.model.save_pretrained(str(saved))
        tok.save_pretrained(str(saved))
    val_row = next(r for r in rows if r["eval_set"] == "val")
    del trainer, model, train_ds
    gc.collect(); torch.cuda.empty_cache()
    return val_row, sec, saved

done = set(pd.read_csv(R07C)["run_id"]) if R07C.exists() else set()
todo = [r for r in RUNS if f"{r[0]}|{r[1]}|{r[2]}|s{r[3]}" not in done]
print(f"07c planned runs: {len(todo)} (already done: {len(RUNS) - len(todo)})")

failed07c, t_list = [], []
for i, (m, meth, ts, s) in enumerate(todo, 1):
    rid = f"{m}|{meth}|{ts}|s{s}"
    eta = (np.mean(t_list) * (len(todo) - i + 1) / 60) if t_list else float("nan")
    print(f"\n[{i}/{len(todo)}] {rid} | ETA ~{eta:.0f} min")
    try:
        vr, sec, saved = train_and_save(m, meth, ts, s)
        t_list.append(sec)
        print(f"   done in {sec / 60:.1f} min | val AUC={vr['auc']:.3f}" + (f" | saved: {saved}" if saved else ""))
    except Exception as e:
        print(f"   ❌ FAILED {rid}: {repr(e)[:300]}")
        if not failed07c:
            print(traceback.format_exc()[-1200:])
        failed07c.append(rid)
        gc.collect(); torch.cuda.empty_cache()
if failed07c:
    print("\nFailed:", failed07c)

# ---------------- summary: 07b (seed 42) + 07c, final config only ----------------
assert R07C.exists(), "Koi 07c run complete nahi hua — upar ke errors dekho."
allr = pd.concat([pd.read_csv(OUT_DIR / "results.csv").query("train_set == 'mbib+babe'"),
                  pd.read_csv(R07C)]).drop_duplicates(["run_id", "eval_set", "group"])
KEY2 = ["mbib_en_test", "babe_test_en", "babe_test_hi", "gold_conf2"]
sub = allr[(allr["group"] == "all") & allr["eval_set"].isin(KEY2)]
fmt = lambda x: f"{x.mean():.3f}" + (f"±{x.std():.3f}" if len(x) > 1 else "")

print("\n" + "=" * 70 + "\nFINAL CONFIG (mbib+babe): mean±std over seeds\n" + "=" * 70)
print("Seeds per config:", sub.groupby(["model", "method"])["seed"].nunique().to_dict())
for metric in ["auc", "f1", "ece"]:
    print(f"\n--- {metric.upper()} ---")
    display(sub.groupby(["model", "method", "eval_set"])[metric].apply(fmt).unstack("eval_set")[KEY2])

print("\nAdapters saved in:", FINAL_DIR)
for p in sorted(FINAL_DIR.glob("*")):
    print("  ", p.name)
print("""
NEXT: Notebook 08 — temperature scaling (val pe fit), Integrated Gradients vs BABE 'biased_words'
(expert word-level annotations => explainability ka human-free evaluation), robustness.
""")